# Task A1 — total event count

Run a JSONiq query (via the RumbleDB-backed `jsoniq` package) that returns the total
number of rows (events) in the data file. The query strictly uses `json-file()` and
`count()`, and execution is wrapped in a timer to report wall-clock runtime in seconds.

In [1]:
# --- Cell 1: environment fix + imports ---
#
# This machine has an external Spark at C:\spark\spark-3.5.1-bin-hadoop3 on PYTHONPATH.
# That Spark ships antlr4-runtime-4.9.3.jar, but RumbleDB 2.1.1 (bundled with `jsoniq`)
# needs antlr4-runtime-4.13.1.jar — which the pip-installed pyspark provides. If the
# external Spark wins, EVERY JSONiq query fails to parse with:
#   InvalidClassException: ATN ... Could not deserialize ATN with version 4 (expected 3)
#
# Fix: before importing pyspark/jsoniq, drop the external Spark from the import path and
# point SPARK_HOME at the pip-installed pyspark. Must run BEFORE the first pyspark import.
import os, sys, sysconfig

os.environ.pop("PYTHONPATH", None)
sys.path[:] = [p for p in sys.path if "spark-3.5.1-bin-hadoop3" not in p]
os.environ["SPARK_HOME"] = os.path.join(sysconfig.get_paths()["purelib"], "pyspark")

import jsoniq          # RumbleDB JSONiq engine for Python
import time            # standard library, for measuring wall-clock time

print("jsoniq imported; SPARK_HOME =", os.environ["SPARK_HOME"])

[Info] Using RumbleDB jar file at: c:\Users\daniel pilant\AppData\Local\Programs\Python\Python311\Lib\site-packages\jsoniq\jars\rumbledb-2.1.1.jar
jsoniq imported; SPARK_HOME = c:\Users\daniel pilant\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark


In [2]:
# --- Cell 2: run the A1 JSONiq query and time it ---
from jsoniq import RumbleSession

# Absolute path to the data file. Forward slashes avoid backslash-escaping inside JSONiq.
DATA_PATH = "C:/Users/daniel pilant/folders/study_material/Year_3_Semester_2/BigData/Week6/data/git-archive-huge.json"

# Fail fast with a clear message if the file is not where we expect it.
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Data file not found:\n  {DATA_PATH}\n"
                            "Update DATA_PATH to the actual file in the data/ folder.")

# Start (or reuse) the RumbleDB session.
rumble = RumbleSession.builder.appName("A1-count").getOrCreate()

# JSONiq query for Task A1:
#   json-file(...) streams the JSONL file as a sequence of objects (gzip auto-detected by extension)
#   count(...)     returns the number of items in that sequence = total events/rows
query = f'count(json-file("{DATA_PATH}"))'

# Time only the query execution (wall-clock).
start = time.perf_counter()
result = rumble.jsoniq(query).json()   # .json() -> Python value; count() yields a 1-item sequence
elapsed = time.perf_counter() - start

# count() returns a single-item sequence, so .json() gives a tuple like (N,); take the scalar.
total_events = result[0] if isinstance(result, (list, tuple)) else result

print(f"Total events (rows): {total_events}")
print(f"A1 wall-clock time: {elapsed:.2f} seconds")

Total events (rows): 28506909
A1 wall-clock time: 127.49 seconds


## Task A2 — event type frequencies

Group every distinct `e.type` with its event count, sorted **descending**, using a FLWOR
`for / group by / return`. Each result object has the shape `{"type": "PushEvent", "count": 1234}`.

In [3]:
# --- Cell 3: A2 — event type frequencies ---
import json  # print each result object as JSON: {"type": ..., "count": ...}

# A2: group every distinct event type (e.type) with its event count, in DESCENDING order.
# Uses a FLWOR expression: for / group by / return. Each returned object has the required
# shape {"type": ..., "count": ...}. Reuses rumble, DATA_PATH and time from the A1 cells.
query_a2 = f'''
for $e in json-file("{DATA_PATH}")
group by $t := $e.type
order by count($e) descending
return {{ "type": $t, "count": count($e) }}
'''

# RumbleDB returns at most 10 result items by default; A2 has 14 event types, so raise the
# result-size cap before fetching — otherwise json() raises CannotMaterializeException.
rumble.getRumbleConf().setResultSizeCap(1_000_000)

# Time only the query execution (wall-clock), same as A1.
start = time.perf_counter()
result_a2 = rumble.jsoniq(query_a2).json()   # tuple of {"type":.., "count":..} objects
elapsed_a2 = time.perf_counter() - start

print(f"Distinct event types: {len(result_a2)}\n")
for obj in result_a2:
    print(json.dumps(obj))
print(f"\nA2 wall-clock time: {elapsed_a2:.2f} seconds")


Distinct event types: 14

{"type": "PushEvent", "count": 14271557}
{"type": "CreateEvent", "count": 3328235}
{"type": "IssueCommentEvent", "count": 2782424}
{"type": "WatchEvent", "count": 2552211}
{"type": "IssuesEvent", "count": 1463914}
{"type": "PullRequestEvent", "count": 1393766}
{"type": "ForkEvent", "count": 971107}
{"type": "DeleteEvent", "count": 548218}
{"type": "PullRequestReviewCommentEvent", "count": 441408}
{"type": "GollumEvent", "count": 294042}
{"type": "CommitCommentEvent", "count": 193894}
{"type": "MemberEvent", "count": 147596}
{"type": "ReleaseEvent", "count": 89280}
{"type": "PublicEvent", "count": 29257}

A2 wall-clock time: 245.80 seconds


## Task A3 — first 5 PushEvents (search in nested fields)

Return `actor.login`, `repo.name` and `created_at` for the **first five PushEvent** records,
using the positional variable `$pos`. We filter to PushEvent first, then keep `$pos le 5`, so
the positions count among PushEvents — i.e. the first five PushEvents.

In [4]:
# --- Cell 5: A3 — first 5 PushEvents (projection of nested fields) ---
import json

# A3: return actor.login, repo.name and created_at for the FIRST FIVE PushEvent records.
# We filter to PushEvent FIRST, then number the survivors with $pos and keep $pos le 5,
# so $pos counts position *among PushEvents* -> the first five PushEvents (the task's goal).
# (The assignment hint applies $pos to the whole sequence; on interleaved data that returns
#  only the PushEvents within the first 5 events overall — usually fewer than 5.)
query_a3 = f'''
for $e at $pos in (
    for $x in json-file("{DATA_PATH}")
    where $x.type eq "PushEvent"
    return $x
)
where $pos le 5
return {{ "actor_login": $e.actor.login, "repo_name": $e.repo.name, "created_at": $e.created_at }}
'''

start = time.perf_counter()
result_a3 = rumble.jsoniq(query_a3).json()
elapsed_a3 = time.perf_counter() - start

print(f"First {len(result_a3)} PushEvents:\n")
for obj in result_a3:
    print(json.dumps(obj))
print(f"\nA3 wall-clock time: {elapsed_a3:.2f} seconds")


First 5 PushEvents:

{"actor_login": "davidcarlsonberg", "repo_name": "PubWlkr/PubWlkr", "created_at": "2015-02-20T01:00:01Z"}
{"actor_login": "loomchild", "repo_name": "loomchild/reload", "created_at": "2015-02-20T01:00:01Z"}
{"actor_login": "lsqshr", "repo_name": "lsqshr/nipype", "created_at": "2015-02-20T01:00:01Z"}
{"actor_login": "PhancyCat", "repo_name": "PhancyCat/HTMLClock", "created_at": "2015-02-20T01:00:01Z"}
{"actor_login": "saramartinez", "repo_name": "saramartinez/tv-or-not-tv", "created_at": "2015-02-20T01:00:01Z"}

A3 wall-clock time: 303.96 seconds


## Task A4 — PushEvents with a non-empty commits array

Count PushEvent records that include a **non-empty** `payload.commits` array (careful with
missing fields), and compare to the total PushEvent count from A2. The unboxing `commits[]`
makes both *missing* and *empty `[]`* fall away, so only pushes with real commits are counted.

In [5]:
# --- Cell 7: A4 — PushEvents with a non-empty commits array ---

# A4: count PushEvent records whose payload.commits array EXISTS and is NON-EMPTY,
# being careful about missing fields. The trick is UNBOXING the array with [] :
#   exists($e.payload.commits[])  is false when commits is missing OR an empty array [],
#   and true only when it has >=1 element. (Plain count($e.payload.commits) would be 1 even
#   for [], wrongly counting empty pushes — the gotcha.)
query_a4 = f'''
count(
    for $e in json-file("{DATA_PATH}")
    where $e.type eq "PushEvent" and exists($e.payload.commits[])
    return $e
)
'''

start = time.perf_counter()
result_a4 = rumble.jsoniq(query_a4).json()   # -> (N,)
elapsed_a4 = time.perf_counter() - start
push_with_commits = result_a4[0] if isinstance(result_a4, (list, tuple)) else result_a4

# Compare to the total PushEvent count from A2 (result_a2 lives in the kernel from the A2 cell).
push_total = next(o["count"] for o in result_a2 if o["type"] == "PushEvent")
empty = push_total - push_with_commits
pct = 100 * push_with_commits / push_total

print(f"PushEvents with non-empty commits: {push_with_commits}")
print(f"Total PushEvents (from A2): {push_total}")
print(f"PushEvents with empty or missing commits: {empty}")
print(f"Share with non-empty commits: {pct:.2f}%")
print(f"\nA4 wall-clock time: {elapsed_a4:.2f} seconds")


PushEvents with non-empty commits: 14173075
Total PushEvents (from A2): 14271557
PushEvents with empty or missing commits: 98482
Share with non-empty commits: 99.31%

A4 wall-clock time: 187.90 seconds


## Task A5 — timestamp range per event type

Return a **single** JSON object keyed by event type, each value `{"earliest", "latest"}` = the
min/max `created_at` for that type. ISO-8601 strings sort lexicographically, so `min`/`max` compare
them correctly. JSONiq projects each event to `{type, ts}`; Spark's `groupBy().agg(min, max)` then
aggregates memory-efficiently (RumbleDB's own group-by min/max OOMs on the 14M-row PushEvent group).

In [6]:
# --- Cell 9: A5 — timestamp range per event type ---
import json
from pyspark.sql import functions as F

# A5: return a SINGLE JSON object keyed by event type, each value {"earliest", "latest"} =
# min/max of created_at for that type. ISO-8601 strings sort lexicographically, so min/max
# compare them correctly.
#
# Why hybrid: RumbleDB's native min/max group-by holds each type's whole group in heap and
# OOMs on the 14M-row PushEvent group. So JSONiq just PROJECTS each event to {type, ts}; we
# hand that DataFrame (.df()) to Spark, whose groupBy().agg() folds to a running min/max
# (memory-efficient) and scales fine within the default heap.
query_a5 = 'for $e in json-file("' + DATA_PATH + '") return { "type": $e.type, "ts": $e.created_at }'

start = time.perf_counter()
df_a5 = rumble.jsoniq(query_a5).df()
rows_a5 = (df_a5.groupBy("type")
                .agg(F.min("ts").alias("earliest"), F.max("ts").alias("latest"))
                .collect())
elapsed_a5 = time.perf_counter() - start

# Assemble the single object keyed by type.
obj_a5 = {row["type"]: {"earliest": row["earliest"], "latest": row["latest"]} for row in rows_a5}

print(json.dumps(obj_a5, indent=2, sort_keys=True))
print(f"\nA5 wall-clock time: {elapsed_a5:.2f} seconds")


{
  "CommitCommentEvent": {
    "earliest": "2015-01-01T00:00:55Z",
    "latest": "2015-02-28T23:58:56Z"
  },
  "CreateEvent": {
    "earliest": "2015-01-01T00:00:01Z",
    "latest": "2015-02-28T23:59:59Z"
  },
  "DeleteEvent": {
    "earliest": "2015-01-01T00:00:30Z",
    "latest": "2015-02-28T23:59:46Z"
  },
  "ForkEvent": {
    "earliest": "2015-01-01T00:00:16Z",
    "latest": "2015-02-28T23:59:34Z"
  },
  "GollumEvent": {
    "earliest": "2015-01-01T00:01:10Z",
    "latest": "2015-02-28T23:59:44Z"
  },
  "IssueCommentEvent": {
    "earliest": "2015-01-01T00:00:06Z",
    "latest": "2015-02-28T23:59:59Z"
  },
  "IssuesEvent": {
    "earliest": "2015-01-01T00:00:30Z",
    "latest": "2015-02-28T23:59:59Z"
  },
  "MemberEvent": {
    "earliest": "2015-01-01T00:04:11Z",
    "latest": "2015-02-28T23:58:56Z"
  },
  "PublicEvent": {
    "earliest": "2015-01-01T00:09:13Z",
    "latest": "2015-02-28T23:57:28Z"
  },
  "PullRequestEvent": {
    "earliest": "2015-01-01T00:00:11Z",
    "latest": 